# 1강: 워드 임베딩과 순환신경망 기반 모델

# 1. 워드 임베딩

# 1-1. 원-핫 인코딩

원-핫 인코딩이란?
- 규칙 기반 혹은 통계적 자연어 처리 연구의 대다수는 단어를 원자적(쪼갤 수 없는) 기호로 취급
  - 예: hotel, conference, walk 같은 단어들
- 벡터 공간 관점에서 보면, 이는 한 원소만 1이고 나머지는 모두 0인 벡터를 의미한다.
  - [00000000000000010000]
- 차원 수(= 단어 사전 크기)는 대략 다음과 같다:
  - 음성 데이터(2만) - Penn Treebank(PTB) 코퍼스(5만) - big vocab(50만) - Google 1T(1300만)
- 이를 원-핫(one-hot) 표현이라고 부르고, 단어를 원-핫 표현으로 바꾸는 과정을 원-핫 인코딩이라고 한다.

원 핫 인코딩의 문제점
- 예: 웹 검색
  - [삼성 노트북 배터리 사이즈] == [삼성 노트북 배터리 용량]
  - [갤럭시 핸드폰] == [갤럭시 스마트폰]
  - 핸드폰 [00000000000000000100000]
  - 스마트폰 [0000100000000000000000] = 0
- 검색 쿼리 벡터와 대상이 되는 문서 벡터들이 서로 직교하게 되어, 원-핫 벡터로는 유사도를 측정할 수 없다.
- 원-핫 인코딩과 같은 전통적인 텍스트 표현 방식에는 여러 한계가 존재한다.
  - 1. 차원의 저주(Curse of Dimensionality)
    - 고차원의 희소 벡터를 다루기 위해선 많은 메모리가 필요하다.
    - 차원이 커질수록 데이터가 점점 더 희소해져서 활용이 어렵다.
  - 2. 의미적 정보 부족
    - 비슷한 단어라도 유사한 벡터로 표현되지 않는다.
    - 예: '은행'과 '금융'은 의미적으로 밀접하지만, 원-핫 인코딩에서는 전혀 무관한 벡터로 취급된다.

# 1-2. 워드 임베딩

주변 단어들을 활용해보기
- 단어를 주변 단어들로 표현하면, 많은 의미르 담을 수 있다.
- 현대 통계적 자연어처리에서의 가장 성공적인 아이디어 중 하나이다.

워드 임베딩이란?
- 단어를 단어들 사이의 의미적 관계를 포착할 수 있는 밀집(dense)되고, 연속적 / 분산적(distributed) 벡터 표현으로 나타내는 방법이다.
  - 원-핫 인코딩에선 은행과 금융이 완전히 독립적인(무관한) 벡터로 표현되었지만, 
  - 워드 임베딩에선 두 단어의 벡터가 공간상 서로 가깝게 위치하며, 이를 통해 의미적 유사성을 반영할 수 있다.

대표적인 워드 임베딩 기법 - Word2Vec
- Word2Vec은 2013년 Google에서 개발한 워드 임베딩 기법
- 단어의 표현을 간단한 인공 신경망을 이용해서 학습

Word2Vec의 아이디어
- Word2Vec의 아이디어는 각 단어와 그 주변 단어들 간의 관계를 예측한다는 것이다.
- Word2Vec엔 두 가지 알고리즘이 존재한다.
  - Skip-grams(SG) 방식
    - 중심 단어를 통해 주변 단어들을 예측하는 방법이다.
    - 단어의 위치(앞 / 뒤)에 크게 구애 받지 않는다.
  - Continuous Bag of Words(CBOW) 방식
    - 주변 단어들을 통해 중심 단어를 예측하는 방법이다.
    - 문맥 단어들의 집합으로 중심 단어를 맞춘다.

Skip-grams(SG): 중심 단어를 통해 주변 단어 예측하기
- 윈도우 크기(window size) = 중심 단어 주변 몇 개 단어를 문맥으로 볼 것인가?
- 예: (윈도우 크기 = 2)
  - 문장: '...problems turning into banking crises as ...'
  - 중심 단어 'banking' (위치 t)
  - 주변 단어 = {'turning', 'into', 'crises', 'as'}

Continuous Bag of Words(CBOW): 주변 단어를 통해 중심 단어 예측하기
- 목표: 주변 단어들의 집합이 주어졌을 때, 그 문맥과 함께 등장할 수 있는 단일 단어를 예측한다.

Skip-Gram
- 장점
  - 적은 데이터에도 잘 동작한다.
  - 희귀 단어나 구 표현에 강하다
- 단점
  - 학습 속도가 느리다

CBOW
- 장점
  - 학습 속도가 빠르다
  - 자주 나오는 단어에 강하다
- 단점
  - 희귀 단어 표현에 약하다

# 2. 순차적 데이터

순차적 데이터란 무엇인가?
- 자연엔 수 많은 순차적 데이터(Sequential Data)가 존재한다.
- 특징
  - 1. 순서가 중요하다.
    - 데이터의 순서가 바뀌면 의미가 달라진다.
    - 예: 나는 너를 사랑해 /= 너는 나를 사랑해
  - 2. 장기 의존성(Long-term dependency)
    - 멀리 떨어진 과거의 정보가 현재 / 미래에 영향을 준다
    - 예: 여러 개의 문 중 파란 문을 열고 안으로 들어가면 너는 (?)를 찾게 될거야
  - 3. 가변 길이(Variable length)
    - 순차 데이터는 길이가 일정하지 않고, 단어 수도 제각각이다.

순차적 데이터를 처리하려면?
- 따라서, 순차적 데이터를 처리하려면 일반적인 모델들(예: 선형회귀, MLP 등)로는 불가능하다.
- Sequential Models이 필요하다.
  - 예: RNN, LSTM, Transformer 등 

# 3. RNN

전통적인 인공신경망
- 전통적인 인공신경망(MLP, CNN)들은 고정된 길이의 압력을 받아 가변 길이의 데이터를 처리하기에 적합하지 않다.

RNN이란?
- 하지만, RNN은 가변 길이의 입력을 받을 수 있고, 이전 입력을 기억할 수 있기 때문에, 순차적 데이터 처리에 적합한 아키텍처이다.

RNN 아키텍처 설명
- 전통적인 신경망(MLP, CNN등)과 달리, RNN은 이전 시점의 정보를 담는 hidden state를 가지고 있다.
- 따라서, 입력 시퀀스 벡터 x를 처리할 때, 각 시점마다 recurrence 수식을 적용하여 hidden state를 업데이트 한다.

RNN의 특징
- RNN은 한 번에 하나의 요소를 처리하고, 정보를 앞으로 전달한다.
- 펼쳐서 보면, RNN은 각 층이 하나의 시점을 나타내는 깊은 신경망처럼 보인다.
- RNN은 hidden state를 유지하면서 가변 길이 데이터를 처리할 수 있다.
- RNN의 출력은 과거 입력에 영향을 받는다는 점에서, feedforward 신경망과 다르다.

RNN의 한계: 기울기 소실(vanishing gradient) 문제
- 기울기 소실 문제란?
  - 딥러닝에서 역전파 시 앞쪽 층의 기울기가 0에 가까워져서 장기 의존성 학습이 어려워 지는 현상
- 왜 일어날까?
 - 1. 역전파 과정에서 작은 값들이 계속 곱해진다.
 - 2. 과거 시점에서 온 오차 신호는 갈수록 더 작은 기울기를 갖는다.
 - 3. 결국 파라미터들이 장기 의존성은 학습하지 못하고, 단기 의존성만 포착하게 된다.